In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import median_abs_deviation
from scipy import sparse
from upsetplot import UpSet, from_indicators
import matplotlib.pyplot as plt
import warnings

In [ ]:
adata = sc.read("../results/adata/06-doublets.h5ad")


In [ ]:
adata

In [ ]:
original_cell_number = adata.n_obs
original_gene_number = adata.n_vars

#### Use rounded soupx counts

In [ ]:
adata.X = adata.layers["soupx_rounded"]

#### Cells per Day

In [ ]:
adata.obs["day"].value_counts()

#### Cells per Sample

In [ ]:
adata.obs["sample"].value_counts()

---
# Gene Annotation

#### Annotate mitochondrial genes

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("mt-")

In [ ]:
adata.var["mt"].value_counts()

#### Annotate ribosomal genes

In [ ]:
adata.var["ribo"] = adata.var_names.str.startswith(("Rps", "Rpl"))

In [ ]:
adata.var["ribo"].value_counts()

#### Annotate hemoglobin genes

In [ ]:
adata.var["hb"] = adata.var_names.str.contains("^Hb[^(P)]")

In [ ]:
adata.var["hb"].value_counts()

---
# Quality metrics for raw data

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb",], percent_top=[20], inplace=True, log1p=True)

In [ ]:
adata

### Malat1 Expression

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata.obs["malat1_expression"] = adata[:, "Malat1"].X.toarray().flatten()

### Intronic Fraction

In [ ]:
def get_counts(layer):
    arr = adata.layers[layer]
    return np.asarray(arr.sum(axis=1)).flatten() if sparse.issparse(arr) else arr.sum(axis=1)

spliced = get_counts('spliced')
unspliced = get_counts('unspliced')

total = spliced + unspliced

# avoid divide-by-zero for empty cells
adata.obs['intronic_fraction'] = np.divide(
    unspliced, total, out=np.zeros_like(total, dtype=float), where=total > 0
)

#### Minimum total count

In [ ]:
int(adata.obs["total_counts"].min())

In [ ]:
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_in_top_20_genes"], groupby="sample", rotation=45)

In [ ]:
sc.pl.violin(adata, ["malat1_expression", "intronic_fraction"], groupby="sample", rotation=45)

In [ ]:
sc.pl.violin(adata, ["pct_counts_mt", "pct_counts_ribo", "pct_counts_hb"],groupby="sample", rotation=45)

In [ ]:
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts", color="pct_counts_mt")

In [ ]:
warnings.filterwarnings("ignore", category=PendingDeprecationWarning)

sc.pl.highest_expr_genes(adata, n_top=20)

warnings.resetwarnings()

---
# Filtering

## Filter cells with low number of genes

In [ ]:
print(f"Number of cells before filtering: {original_cell_number}")
sc.pp.filter_cells(adata, min_genes=500)
print(f"Number of cells after filtering: {adata.n_obs}")

## Function to compute median absolute deviation outliers

In [ ]:
def is_outlier(adata, metric: str, nmads: int, direction="both"):
    M = adata.obs[metric]
    med = np.median(M)
    mad = median_abs_deviation(M)

    if direction == "high":
        return M > med + nmads * mad
    elif direction == "low":
        return M < med - nmads * mad
    elif direction == "both":
        return (M < med - nmads * mad) | (M > med + nmads * mad)
    else:
        raise ValueError("direction must be 'high', 'low', or 'both'")

In [ ]:
def mad_outliers_by_sample(adata, sample_col, metric_col, n_mads, direction="both"):
    """
    Flag MAD-based outliers for a QC metric, computed independently within each sample.

    For each group in `adata.obs[sample_col]`, computes the median and median
    absolute deviation (MAD) of `adata.obs[metric_col]`, then flags cells that
    fall outside `n_mads` MADs from that group's own median.

    Parameters
    ----------
    adata : AnnData
        Annotated data object; metric and sample columns are read from adata.obs.
    sample_col : str
        Column in adata.obs identifying the sample/batch to group by.
    metric_col : str
        Column in adata.obs containing the QC metric to evaluate.
    n_mads : float
        Number of MADs from the median beyond which a value is considered an outlier.
    direction : {"both", "upper", "lower"}, default "both"
        - "both":  flag values below (median - n_mads*MAD) OR above (median + n_mads*MAD)
        - "upper": flag values above (median + n_mads*MAD) only
        - "lower": flag values below (median - n_mads*MAD) only

    Returns
    -------
    pd.Series
        Boolean Series aligned to adata.obs.index, True where the cell is an
        outlier for metric_col relative to its own sample's median/MAD.
    """
    if direction not in ("both", "upper", "lower"):
        raise ValueError(f"direction must be 'both', 'upper', or 'lower', got {direction!r}")

    values = adata.obs[metric_col]
    groups = adata.obs.groupby(sample_col, observed=True)[metric_col]

    median = groups.transform("median")
    mad = groups.transform(lambda x: median_abs_deviation(x, nan_policy="omit"))

    lower_bound = median - n_mads * mad
    upper_bound = median + n_mads * mad

    if direction == "both":
        outlier = (values < lower_bound) | (values > upper_bound)
    elif direction == "upper":
        outlier = values > upper_bound
    else:  # "lower"
        outlier = values < lower_bound

    # If a sample's MAD is 0 don't flag anything
    outlier = outlier & (mad > 0)

    return outlier.reindex(adata.obs.index)

## Identify count outliers
- log1p_total_counts: Log of the total number of counts for a cell
- log1p_n_genes_by_counts: The number of genes with at least 1 count in a cell.
- pct_counts_in_top_20_genes: Cumulative percentage of counts for the 20 most expressed genes

In [ ]:
adata.obs["log1p_total_counts_outlier"] =         mad_outliers_by_sample(adata, "sample", "log1p_total_counts", 3)
adata.obs["log1p_n_genes_by_counts_outlier"] =    mad_outliers_by_sample(adata, "sample", "log1p_n_genes_by_counts", 3)
adata.obs["pct_counts_in_top_20_genes_outlier"] = mad_outliers_by_sample(adata, "sample", "pct_counts_in_top_20_genes", 3)
adata.obs["pct_counts_mt_outlier"] =              mad_outliers_by_sample(adata, "sample", "pct_counts_mt", 3)
adata.obs["malat1_expression_outlier"] =          mad_outliers_by_sample(adata, "sample", "malat1_expression", 3)
adata.obs["intronic_fraction_outlier"] =          mad_outliers_by_sample(adata, "sample", "intronic_fraction", 3)

## Set hard hemoglobin gene filter

In [ ]:
adata.obs["pct_counts_hb_outlier"] = adata.obs["pct_counts_hb"] > 0.5

## Filter

In [ ]:
outlier_cols = [
    "log1p_total_counts_outlier",
    "log1p_n_genes_by_counts_outlier",
    "pct_counts_in_top_20_genes_outlier",
    "pct_counts_mt_outlier",
    "malat1_expression_outlier",
    "pct_counts_hb_outlier",
    "intronic_fraction_outlier"
]

warnings.filterwarnings("ignore", category=FutureWarning)

upset_data = from_indicators(outlier_cols, adata.obs[outlier_cols])
UpSet(upset_data, sort_by="cardinality").plot()
plt.show()

warnings.resetwarnings()

In [ ]:
# True if any outlier flag is True
outlier_mask = adata.obs[outlier_cols].any(axis=1)

# Keep only non-outliers
adata_filtered = adata[~outlier_mask].copy()

In [ ]:
print(f"Original number of cells: {original_cell_number}")
print(f"Number of cells after filtering: {adata_filtered.n_obs}")

print(f"Original number of genes: {original_gene_number}")
print(f"Number of genes after filtering: {adata_filtered.n_vars}")


In [ ]:
adata_filtered.obs["sample"].value_counts()

## Plot filtered quality metrics

In [ ]:
sc.pl.violin(adata_filtered, ["n_genes_by_counts", "total_counts", "pct_counts_in_top_20_genes"], groupby="sample", rotation=45)

In [ ]:
sc.pl.violin(adata_filtered, ["malat1_expression", "intronic_fraction"], groupby="sample", rotation=45)

In [ ]:
sc.pl.violin(adata_filtered, ["pct_counts_mt", "pct_counts_ribo", "pct_counts_hb"], groupby="sample", rotation=45)

In [ ]:
sc.pl.scatter(adata_filtered, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

## Doublets

In [ ]:
adata.obs["scdblfinder_class"].value_counts()

In [ ]:
doublets = adata[adata.obs["scdblfinder_class"] != 1]

### Cells flagged as doublets

In [ ]:
sc.pl.violin(doublets, ["n_genes_by_counts", "total_counts"], groupby="sample", rotation=45)

In [ ]:
sc.pl.scatter(
    adata,
    x="total_counts",
    y="n_genes_by_counts",
    color="scdblfinder_class"
)

### Filtered

In [ ]:
sc.pl.violin(adata_filtered, ["n_genes_by_counts", "total_counts"], groupby="sample", rotation=45)

In [ ]:
adata_filtered.write("../results/adata/07-filter.h5ad")